In [ ]:
import rasterio
from rasterio.windows import Window
import numpy as np
import cupy as cp 
from tqdm import tqdm
import time

TILE_SIZE = 1024
BATCH_SIZE = 16 
CHANNELS = 2    

def process_batch_gpu(batch_data):
    
    data_gpu = cp.asarray(batch_data)
    
    red = data_gpu[:, 0, :, :]
    nir = data_gpu[:, 1, :, :]
    
    ndvi_gpu = (nir - red) / (nir + red + 1e-6)
    
    return cp.asnumpy(ndvi_gpu)

def main():
    red_path = r'C:\Users\hsgpi\OneDrive\Desktop\S2C_MSIL2A_20260213T045921_N0512_R119_T44QQE_20260213T133817.SAFE\GRANULE\L2A_T44QQE_A007525_20260213T051019\IMG_DATA\R10m\B04_10m.jp2'
    nir_path = r'C:\Users\hsgpi\OneDrive\Desktop\S2C_MSIL2A_20260213T045921_N0512_R119_T44QQE_20260213T133817.SAFE\GRANULE\L2A_T44QQE_A007525_20260213T051019\IMG_DATA\R10m\B08_10m.jp2'
    output_path = 'batch_ndvi_output.tif'
    
    start_time = time.time()

    with rasterio.open(red_path) as red_src, rasterio.open(nir_path) as nir_src:
        meta = red_src.meta.copy()
        meta.update(driver='GTiff', dtype='float32', count=1, tiled=True, compress='lzw')
        
        h, w = red_src.height, red_src.width
        
        windows = []
        for r in range(0, h, TILE_SIZE):
            for c in range(0, w, TILE_SIZE):
                windows.append(Window(c, r, min(TILE_SIZE, w-c), min(TILE_SIZE, h-r)))
        
        with rasterio.open(output_path, 'w', **meta) as dst:
            for i in tqdm(range(0, len(windows), BATCH_SIZE)):
                batch_windows = windows[i : i + BATCH_SIZE]
                
                batch_array = np.zeros((len(batch_windows), CHANNELS, TILE_SIZE, TILE_SIZE), dtype='float32')
                
                for idx, win in enumerate(batch_windows):
                    r_tile = red_src.read(1, window=win, boundless=True, fill_value=0)
                    n_tile = nir_src.read(1, window=win, boundless=True, fill_value=0)
                    batch_array[idx, 0, :win.height, :win.width] = r_tile
                    batch_array[idx, 1, :win.height, :win.width] = n_tile
                
                batch_ndvi = process_batch_gpu(batch_array)
                
                for idx, win in enumerate(batch_windows):
                    dst.write(batch_ndvi[idx, :win.height, :win.width], 1, window=win)

    print(f"SOGNA Batch Processing Complete: {time.time() - start_time:.2f}s")

if __name__ == "__main__":
    main()

In [ ]:
import rasterio
import os

red_jp2 = r"C:\Users\hsgpi\OneDrive\Desktop\S2C_MSIL2A_20260213T045921_N0512_R119_T44QQE_20260213T133817.SAFE\GRANULE\L2A_T44QQE_A007525_20260213T051019\IMG_DATA\R10m\B04_10m.jp2"
nir_jp2 = r"C:\Users\hsgpi\OneDrive\Desktop\S2C_MSIL2A_20260213T045921_N0512_R119_T44QQE_20260213T133817.SAFE\GRANULE\L2A_T44QQE_A007525_20260213T051019\IMG_DATA\R10m\B08_10m.jp2"

def convert_to_tif(jp2_path):
    
    folder = os.path.dirname(jp2_path)
    name = os.path.splitext(os.path.basename(jp2_path))[0]
    
    tif_path = os.path.join(folder, name + "_converted.tif")

    with rasterio.open(jp2_path) as src:
        profile = src.profile
        profile.update(driver="GTiff", compress="LZW")

        with rasterio.open(tif_path, "w", **profile) as dst:
            for i in range(1, src.count + 1):
                dst.write(src.read(i), i)

    print("Converted:", tif_path)
    return tif_path


red_tif = convert_to_tif(red_jp2)
nir_tif = convert_to_tif(nir_jp2)

In [1]:
import rasterio
from rasterio.windows import Window
import numpy as np
import cupy as cp
from tqdm import tqdm
import time

TILE_SIZE = 256
BATCH_SIZE = 16

red_path = r"C:\Users\hsgpi\OneDrive\Desktop\S2C_MSIL2A_20260213T045921_N0512_R119_T44QQE_20260213T133817.SAFE\GRANULE\L2A_T44QQE_A007525_20260213T051019\IMG_DATA\R10m\B04_10m_converted.tif"
nir_path = r"C:\Users\hsgpi\OneDrive\Desktop\S2C_MSIL2A_20260213T045921_N0512_R119_T44QQE_20260213T133817.SAFE\GRANULE\L2A_T44QQE_A007525_20260213T051019\IMG_DATA\R10m\B08_10m_converted.tif"

output_path = "ndvi_fast.tif"


def compute_ndvi_gpu(batch):

    batch_gpu = cp.asarray(batch)

    red = batch_gpu[:,0,:,:]
    nir = batch_gpu[:,1,:,:]

    ndvi = (nir - red) / (nir + red + 1e-6)

    return cp.asnumpy(ndvi)


def main():

    start = time.time()

    with rasterio.open(red_path) as red_src, rasterio.open(nir_path) as nir_src:

        profile = red_src.profile
        profile.update(
            driver="GTiff",
            dtype="float32",
            count=1,
            compress="LZW",
            tiled=True
        )

        h = red_src.height
        w = red_src.width

        windows = []

        for r in range(0, h, TILE_SIZE):
            for c in range(0, w, TILE_SIZE):

                win = Window(
                    c,
                    r,
                    min(TILE_SIZE, w-c),
                    min(TILE_SIZE, h-r)
                )

                windows.append(win)

        with rasterio.open(output_path,"w",**profile) as dst:

            for i in tqdm(range(0,len(windows),BATCH_SIZE)):

                batch_windows = windows[i:i+BATCH_SIZE]

                batch = np.zeros(
                    (len(batch_windows),2,TILE_SIZE,TILE_SIZE),
                    dtype="float32"
                )

                for idx,win in enumerate(batch_windows):

                    red = red_src.read(1,window=win,boundless=True,fill_value=0)
                    nir = nir_src.read(1,window=win,boundless=True,fill_value=0)

                    batch[idx,0,:win.height,:win.width] = red
                    batch[idx,1,:win.height,:win.width] = nir

                ndvi_batch = compute_ndvi_gpu(batch)

                for idx,win in enumerate(batch_windows):

                    dst.write(
                        ndvi_batch[idx,:win.height,:win.width],
                        1,
                        window=win
                    )

    print("Processing finished in",time.time()-start,"seconds")


if __name__ == "__main__":
    main()

100%|██████████| 116/116 [00:27<00:00,  4.17it/s]


Processing finished in 31.572468280792236 seconds
